In [ ]:
url='https://es.wikipedia.org/wiki/Moci%C3%B3n_de_censura_(Per%C3%BA)'

# coding: utf-8
import pandas as pd
from urllib.parse import quote

# Pull tables from Wikipedia
base  = "https://es.wikipedia.org/wiki/"
path  = "Moci%C3%B3n_de_censura_(Per%C3%BA)"
# url   = base + quote(path)          # Per%C3%BA
censuTables = pd.read_html(url, flavor="bs4",
                      attrs={"class": "wikitable"})

In [ ]:
censuras=censuTables[0].copy()

In [ ]:
consts={1823 : 'Constitución Política de la República Peruana',
1826 : 'Constitución para la República Peruana',
1828 : 'Constitución Política de la República Peruana',
1834 : 'Constitución Política de la República Peruana',
1836 : 'Confederación Perú Boliviana: Constitución del Estado Sud Peruano y del Estado Nor Peruano',
1836 : 'Confederación Perú Boliviana: Decreto del 28 de octubre de 1836',
1837 : 'Ley Fundamental de la Confederación Perú Boliviana',
1839 : 'Constitución Política de la República Peruana',
1856 : 'Constitución de la República Peruana',
1860 : 'Constitución Política del Perú'}
consts

In [ ]:
censuras['constitution']=None
censuras.drop(columns='Ref.',inplace=True)

In [ ]:
censuras['constitution']=censuras.loc[censuras.Fecha.str.contains('Const'),'Fecha']
censuras['constitution']=censuras.constitution.ffill()
censuras

In [ ]:
censuras=censuras.loc[~censuras.Fecha.str.contains('Const'),].reset_index(drop=True)
censuras

In [ ]:
censuras.loc[19,'Fecha']='1 de octubre de 1945'

In [ ]:
import locale

# Set Spanish locale (use es_ES.utf8 or es_ES.UTF-8 depending on OS) - windows: "Spanish_Spain" or "Spanish_Spain.1252"
locale.setlocale(locale.LC_TIME, "es_ES.UTF-8")

from datetime import datetime

spanish_cellToGoodDates=lambda x: datetime.strptime(x, "%d de %B de %Y")

censuras['Fecha']=censuras.Fecha.str.replace('setiembre','septiembre')
# Apply parsing with datetime and locale
censuras['dateCensoring']= censuras['Fecha'].map(spanish_cellToGoodDates)

censuras['year']=censuras.dateCensoring.dt.year
censuras


In [ ]:
constName=[]
import numpy

text='Constitucion de '
for i in range(len(censuras)):
    
    if pd.isna(censuras.loc[i,'constitution']):
        y=censuras.loc[i,'year']
        print(y)
        lista=list(consts.keys())
        listok=[l for l in lista if l<y]
        constiok=max(listok)
        censuras.loc[i,'constitution']=text+str(constiok)



In [ ]:
censuras

In [ ]:
censuras.constitution.value_counts()

In [ ]:
pd.crosstab(censuras.Ministerio,censuras.constitution,margins=True)

In [ ]:
import pandas as pd


string_to_keep = 'Consejo de Ministros'
censuras['Ministerio'] = censuras.Ministerio.str.replace(f'(.*?)({string_to_keep})(.*)', r'\2', regex=True)

# print(df)

In [ ]:
censuras['Ministerio'].value_counts()

In [ ]:



replacement_string = 'Interior'
censuras['Ministerio_new']=censuras['Ministerio']
interiorRep = {'Gobierno, Policía y Obras': replacement_string,'Gobierno y Policía': replacement_string, 'Gobierno': replacement_string}
censuras = censuras.replace({'Ministerio_new':interiorRep}, regex=True)
censuras

In [ ]:
replacement_string = 'Educación'
educaRep = {'Educación Pública': replacement_string}
censuras = censuras.replace({'Ministerio_new':educaRep}, regex=True)
censuras

In [ ]:
replacement_string = 'Economia'
ecoRep = {'Relaciones Exteriores y encargado de Hacienda':replacement_string,'Hacienda': replacement_string}
censuras = censuras.replace({'Ministerio_new':ecoRep}, regex=True)
censuras

In [ ]:
replacement_string = 'Agricultura'
agRep = {'Agricultura y Alimentación':replacement_string}
censuras = censuras.replace({'Ministerio_new':agRep}, regex=True)
censuras

In [ ]:
replacement_string = 'Trabajo'
trRep = {'Trabajo y Comunidades':replacement_string,'Trabajo y Promoción del Empleo':replacement_string}
censuras = censuras.replace({'Ministerio_new':trRep}, regex=True)
censuras

In [ ]:
censuras['Ministerio_new'].value_counts()

In [ ]:
pd.crosstab(censuras.Ministerio_new,censuras.constitution,margins=True)

In [ ]:
censuras.to_excel('censuras.xlsx',index=False)

In [ ]:
censuras

In [ ]:
censuras.info()

In [ ]:
censuras['quarterCensoring']=censuras['year'].map(str)+'-'+censuras['dateCensoring'].dt.quarter.map(str)
censuras['quarterCensoring']

In [ ]:
censuras['dateCensoring']

In [ ]:
censuras['quarterCensoring_period']=censuras['dateCensoring'].dt.to_period(freq='Q')
censuras.info()


In [ ]:
def get_split_year(date):
    year = date.year
    if date.month > 7 or (date.month == 7 and date.day >= 28):
        return year
    else:
        return year - 1

censuras['split_year'] = censuras['dateCensoring'].apply(get_split_year)

def get_split_year_identifier(year):
    return f"{year}-{year + 1}"

censuras['split_year_period'] = censuras['split_year'].apply(get_split_year_identifier)
censuras

In [ ]:
censuras.head()

In [ ]:
censuras.columns

In [ ]:
censuras['cases']=1

In [ ]:
censuras.groupby('quarterCensoring_period')['cases'].sum()

In [ ]:
import pandas as pd

# Assuming your DataFrame is named 'censuras' and the quarter column is 'quarterCensoring_period'
# and the cases column is 'cases'

# 1. Determine the full range of quarters
# Convert 'quarterCensoring_period' to Period objects for easier manipulation
censuras['quarter_period'] = pd.PeriodIndex(censuras['quarterCensoring_period'], freq='Q')

# # Find the minimum and maximum quarter
min_quarter = censuras['quarter_period'].min()
max_quarter = censuras['quarter_period'].max()

# # Create a complete range of quarters
all_quarters = pd.period_range(min_quarter, max_quarter, freq='Q')

# # 2. Group by 'quarterCensoring_period' and sum 'cases'
quarterly_cases = censuras.groupby('quarterCensoring_period')['cases'].sum()

# # Convert the index of the grouped result to PeriodIndex for alignment
quarterly_cases.index = pd.PeriodIndex(quarterly_cases.index, freq='Q')

# # 3. Reindex with all quarters and fill missing values with 0
quarterly_cases_complete = quarterly_cases.reindex(all_quarters, fill_value=0)

print(quarterly_cases_complete)

In [ ]:
censoring_inQ=quarterly_cases_complete.reset_index()
censoring_inQ

In [ ]:
censoring_inQ.rename(columns={'index':'quarter'},inplace=True)

In [ ]:
censoring_inQ['quarter_merge']=censoring_inQ['quarter'].astype('str')

In [ ]:
censoring_inQ.info()

In [ ]:
censoring_inQ_model=censoring_inQ[censoring_inQ.quarter>'1979Q4']
censoring_inQ_model.reset_index(drop=True,inplace=True)
censoring_inQ_model

In [ ]:
censoring_inQ_model.to_parquet('censoring_inQ_model.parquet')
censoring_inQ_model.to_excel('censoring_inQ_model.xlsx',index=False)